# 04 — Experiments and champion model

## What is this notebook for?

When a score is weak, changing features, algorithms and hyperparameters all at once makes it impossible to know what helped.

So we use **controlled ablation**:

> Add one signal layer at a time, keep the comparison fair, and let future-like validation decide.

In [ ]:
import sys
from pathlib import Path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
print("Project root:", project_root)
import warnings
warnings.filterwarnings("ignore")

from src.config import data_dir
from src.data.load import load_source_tables, build_master_table
from src.features.build import build_offline_features
from src.models.train import chronological_split, fit_model, predict_proba
from src.models.experiments import run_experiments, choose_champion
from src.models.evaluate import classification_metrics
from src.models.specs import EXPERIMENTS

Project root: /mnt/data/northbridge-sla-breach-work/northbridge-sla-breach


## 1. Build the complete leakage-safe feature table

Notebook 03 explained each layer. Here we use one production function to keep the experiment notebook readable.

In [2]:
tickets, agents, clients = load_source_tables(data_dir())
master = build_master_table(tickets, agents, clients)
features = build_offline_features(master, agents, sla_due_known=True)
print(f"Feature table: {features.shape[0]:,} rows × {features.shape[1]} columns")
features[["TicketID","CreatedAt","CategoryID","Channel","SLAWindowHours",
          "agent_open_backlog","hist_resolution_burden","SLABreached"]].head()

Feature table: 3,500 rows × 64 columns


,TicketID,CreatedAt,CategoryID,Channel,SLAWindowHours,agent_open_backlog,hist_resolution_burden,SLABreached
0,2179,2023-01-01 10:54:00,5,Form,72.000,0.000,NaN,0
1,322,2023-01-01 11:11:00,2,Email,48.000,0.000,NaN,0
2,1331,2023-01-01 20:04:00,4,Portal,12.000,0.000,NaN,0
3,493,2023-01-01 21:11:00,1,Email,72.000,0.000,NaN,0
4,2022,2023-01-04 04:49:00,1,Email,72.000,1.000,NaN,0


## 2. Split chronologically

We want to predict future tickets, so we train on the past.

- Train teaches.
- Validation chooses.
- Test remains sealed until one winner has been selected.

In [3]:
train, validation, test = chronological_split(features)
split_summary = pd.DataFrame([
 ["Train",len(train),train["CreatedAt"].min(),train["CreatedAt"].max(),train["SLABreached"].mean()],
 ["Validation",len(validation),validation["CreatedAt"].min(),validation["CreatedAt"].max(),validation["SLABreached"].mean()],
 ["Test",len(test),test["CreatedAt"].min(),test["CreatedAt"].max(),test["SLABreached"].mean()],
],columns=["Split","Rows","From","To","Breach rate"])
split_summary

,Split,Rows,From,To,Breach rate
0,Train,2450,2023-01-01 10:54:00,2024-07-31 01:19:00,0.211
1,Validation,525,2024-07-31 06:33:00,2024-11-29 01:25:00,0.213
2,Test,525,2024-11-30 04:18:00,2025-03-31 07:56:00,0.240


## 3. The five ablation experiments

| Experiment | New question |
|---|---|
| E1 | Can static T1 context rank risk? |
| E2 | Does SLA/time geometry help? |
| E3 | Does service pressure help? |
| E4 | Does historical burden/risk help? |
| E5 | Do text hints and explicit interactions help? |

The experiments are cumulative, but model settings stay mostly fixed.

In [4]:
experiment_design=[]
for name,spec in EXPERIMENTS.items():
    experiment_design.append({
        "Experiment":name,
        "Categorical":len(spec["categorical"]),
        "Numeric":len(spec["numeric"]),
        "Total":len(spec["categorical"])+len(spec["numeric"])
    })
pd.DataFrame(experiment_design)

,Experiment,Categorical,Numeric,Total
0,E1_baseline,4,7,11
1,E2_sla_time,4,13,17
2,E3_service_pressure,4,22,26
3,E4_historical_signal,4,29,33
4,E5_full_signal,6,34,40


### A few feature names make the design concrete

In [5]:
for name,spec in EXPERIMENTS.items():
    print(f"\n{name}")
    print("  categorical:", spec["categorical"])
    print("  numeric sample:", spec["numeric"][:8], "..." if len(spec["numeric"])>8 else "")


E1_baseline
  categorical: ['CategoryID', 'Channel', 'ContractTier', 'TeamID']
  numeric sample: ['priority_urgency', 'SLACreditClause', 'DailyCapacity', 'created_hour', 'created_day_of_week', 'is_weekend', 'is_after_hours'] 

E2_sla_time
  categorical: ['CategoryID', 'Channel', 'ContractTier', 'TeamID']
  numeric sample: ['priority_urgency', 'SLACreditClause', 'DailyCapacity', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'is_weekend'] ...

E3_service_pressure
  categorical: ['CategoryID', 'Channel', 'ContractTier', 'TeamID']
  numeric sample: ['priority_urgency', 'SLACreditClause', 'DailyCapacity', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'is_weekend'] ...

E4_historical_signal
  categorical: ['CategoryID', 'Channel', 'ContractTier', 'TeamID']
  numeric sample: ['priority_urgency', 'SLACreditClause', 'DailyCapacity', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'is_weekend'] ...

E5_full_signal
  categorical: ['CategoryID', 'Channel', 'ContractTier', 'TeamID', 'category_channe

## 4. Why three classifiers?

**Logistic Regression** — simple additive baseline.

**XGBoost** — nonlinear boosted trees.

**CatBoost** — boosted trees designed for categorical data.

If all three struggle, the problem may be weak information rather than weak algorithms.

## 5. Run 5 feature sets × 3 classifiers

All 15 runs use the same validation period.

In [6]:
results = run_experiments(train, validation, random_state=42)
results.sort_values(["roc_auc","pr_auc"],ascending=False).round(4).reset_index(drop=True)

,experiment,model,n_features,roc_auc,pr_auc,precision_050,recall_050,f1_050,brier,lift_at_10pct
0,E1_baseline,Logistic Regression,11,0.560,0.255,0.237,0.527,0.327,0.247,1.238
1,E5_full_signal,Logistic Regression,40,0.546,0.253,0.224,0.473,0.304,0.248,1.592
2,E2_sla_time,XGBoost,17,0.531,0.242,0.249,0.384,0.302,0.231,0.884
3,E3_service_pressure,Logistic Regression,26,0.529,0.239,0.209,0.330,0.256,0.237,1.061
4,E2_sla_time,Logistic Regression,17,0.526,0.244,0.218,0.482,0.300,0.250,1.238
5,E1_baseline,CatBoost,11,0.523,0.235,0.237,0.375,0.291,0.237,1.238
6,E1_baseline,XGBoost,11,0.522,0.238,0.243,0.366,0.292,0.234,1.327
7,E4_historical_signal,Logistic Regression,33,0.509,0.222,0.209,0.402,0.275,0.246,0.973
8,E2_sla_time,CatBoost,17,0.507,0.242,0.212,0.312,0.253,0.236,1.327
9,E3_service_pressure,XGBoost,26,0.503,0.226,0.221,0.152,0.180,0.206,0.884


## 6. Compare ROC-AUC across the ablation path

In [7]:
(results.pivot(index="experiment",columns="model",values="roc_auc")
 .loc[list(EXPERIMENTS.keys())].round(3))

model,CatBoost,Logistic Regression,XGBoost
experiment,,,
E1_baseline,0.523,0.560,0.522
E2_sla_time,0.507,0.527,0.531
E3_service_pressure,0.487,0.529,0.503
E4_historical_signal,0.464,0.509,0.489
E5_full_signal,0.486,0.546,0.501


And PR-AUC, which focuses more heavily on the breach class:

In [8]:
(results.pivot(index="experiment",columns="model",values="pr_auc")
 .loc[list(EXPERIMENTS.keys())].round(3))

model,CatBoost,Logistic Regression,XGBoost
experiment,,,
E1_baseline,0.235,0.255,0.238
E2_sla_time,0.242,0.244,0.242
E3_service_pressure,0.228,0.239,0.226
E4_historical_signal,0.205,0.222,0.210
E5_full_signal,0.213,0.252,0.217


### What happened?

The simple **E1 Logistic Regression** gives the best validation ROC-AUC.

Several richer feature sets make performance worse.

That is scientifically useful:

> **A business-plausible feature can still be unstable or noisy out of time.**

## 7. Operational metrics beyond ROC-AUC

We also examine:

- Precision: how many alerts are correct?
- Recall: how many actual breaches are found?
- Lift@10%: how much better is the riskiest 10% than random selection?
- Brier score: how accurate are the probabilities?

In [9]:
results.sort_values("lift_at_10pct",ascending=False)[
 ["experiment","model","roc_auc","pr_auc","precision_050","recall_050","lift_at_10pct"]
].head(8).round(3)

,experiment,model,roc_auc,pr_auc,precision_050,recall_050,lift_at_10pct
12,E5_full_signal,Logistic Regression,0.546,0.252,0.224,0.473,1.592
5,E2_sla_time,CatBoost,0.507,0.242,0.212,0.312,1.327
1,E1_baseline,XGBoost,0.522,0.238,0.243,0.366,1.327
2,E1_baseline,CatBoost,0.523,0.235,0.237,0.375,1.238
3,E2_sla_time,Logistic Regression,0.527,0.244,0.218,0.482,1.238
0,E1_baseline,Logistic Regression,0.560,0.255,0.237,0.527,1.238
11,E4_historical_signal,CatBoost,0.464,0.205,0.230,0.179,1.150
8,E3_service_pressure,CatBoost,0.487,0.228,0.237,0.205,1.061


## 8. Select the champion on validation only

In [10]:
champion = choose_champion(results)
champion.to_frame("Champion result")

,Champion result
experiment,E1_baseline
model,Logistic Regression
n_features,11
roc_auc,0.560
pr_auc,0.255
precision_050,0.237
recall_050,0.527
f1_050,0.327
brier,0.247
lift_at_10pct,1.238


## 9. Refit once and unlock the future test

Only after selection do we combine Train + Validation, fit the chosen architecture, and evaluate the held-out Test period.

In [11]:
train_validation = pd.concat([train,validation]).sort_values(["CreatedAt","TicketID"])
spec = EXPERIMENTS[champion["experiment"]]
final_model = fit_model(champion["model"],train_validation,
                        spec["categorical"],spec["numeric"],random_state=42)
test_probability = predict_proba(final_model,test)
test_metrics = classification_metrics(test["SLABreached"],test_probability)

pd.DataFrame([
 ["Validation",champion["roc_auc"],champion["pr_auc"],champion["lift_at_10pct"],champion["brier"]],
 ["Future test",test_metrics["roc_auc"],test_metrics["pr_auc"],test_metrics["lift_at_10pct"],test_metrics["brier"]],
],columns=["Period","ROC-AUC","PR-AUC","Lift@10%","Brier"]).round(4)

,Period,ROC-AUC,PR-AUC,Lift@10%,Brier
0,Validation,0.560,0.255,1.238,0.247
1,Future test,0.508,0.260,1.179,0.252


## 10. Interpret the result without hiding it

Validation is modestly above chance, but future-test ROC-AUC falls close to **0.50**.

The correct response is not to keep tuning until the test set looks good.

It is evidence that:

> **The current T1 feature space contains weak and unstable forward-looking SLA-breach signal.**

That conclusion motivates better information architecture rather than more model hopping.

## 11. What should come next?

The next experiments should improve the visibility of the process:

1. TF-IDF or embeddings from the initial ticket description;
2. effective staffed hours before the SLA deadline;
3. queue-ahead features: higher-priority and earlier-due tickets;
4. T2 re-scoring after first response;
5. survival/time-to-resolution modelling.

These are **signal experiments**, not cosmetic algorithm changes.

## 12. Where MLflow belongs

The notebook stays teaching-first.

The production script `scripts/train_champion.py` repeats the experiment, saves the champion artifact and logs to MLflow when MLflow is available.

```text
Notebook -> understand
src/     -> reusable logic
scripts/ -> production execution/tracking
```

### Final lesson

When classifiers plateau:

1. check that the experiment is scientifically clean;
2. check whether the business process is sufficiently visible in the features;
3. only then spend time on another algorithm or deeper tuning.